# 01 read excel

Leitura e exploração do arquivo excel

In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from config import engine
from sqlalchemy import text

print("Conexão pronta para uso.")

In [ ]:
import pandas as pd
EXCEL_PATH = '../data/raw/Sample - Superstore.xls'
# Verificar abas disponíveis no arquivo
xl = pd.ExcelFile(EXCEL_PATH, engine='xlrd')
print('Abas encontradas:', xl.sheet_names)
# Ler cada aba
orders = pd.read_excel(EXCEL_PATH, sheet_name='Orders', engine='xlrd')
returns = pd.read_excel(EXCEL_PATH, sheet_name='Returns', engine='xlrd')

people = pd.read_excel(EXCEL_PATH, sheet_name='People', engine='xlrd')
print(f'Orders : {orders.shape[0]:,} linhas x {orders.shape[1]} colunas')
print(f'Returns: {returns.shape[0]:,} linhas x {returns.shape[1]} colunas')
print(f'People : {people.shape[0]:,} linhas x {people.shape[1]} colunas')
# Inspecionar colunas da aba Orders
print('\nColunas de Orders:')
for col in orders.columns:
    print(f' {col:30} -> {orders[col].dtype}')

Abas encontradas: ['Orders', 'Returns', 'People']
Orders : 9,994 linhas x 21 colunas
Returns: 296 linhas x 2 colunas
People : 4 linhas x 2 colunas

Colunas de Orders:
 Row ID                         -> int64
 Order ID                       -> str
 Order Date                     -> datetime64[us]
 Ship Date                      -> datetime64[us]
 Ship Mode                      -> str
 Customer ID                    -> str
 Customer Name                  -> str
 Segment                        -> str
 Country                        -> str
 City                           -> str
 State                          -> str
 Postal Code                    -> int64
 Region                         -> str
 Product ID                     -> str
 Category                       -> str
 Sub-Category                   -> str
 Product Name                   -> str
 Sales                          -> float64
 Quantity                       -> int64
 Discount                       -> float64
 Profit          

In [ ]:
# Verificar valores nulos
print('=== NULOS POR COLUNA ===')
print(orders.isnull().sum()[orders.isnull().sum() > 0])
# Verificar duplicatas de Order ID + Product ID
dups = orders.duplicated(subset=['Order ID','Product ID'])
print(f'\nLinhas duplicadas (Order+Product): {dups.sum()}')
# Estatísticas das métricas principais
print('\n=== ESTATÍSTICAS DAS MÉTRICAS ===')
print(orders[['Sales','Profit','Discount','Quantity']].describe().round(2))
# Distribuição por Segment e Region
print('\n=== DISTRIBUIÇÃO SEGMENT ===')
print(orders['Segment'].value_counts())
print('\n=== DISTRIBUIÇÃO REGION ===')
print(orders['Region'].value_counts())

=== NULOS POR COLUNA ===
Series([], dtype: int64)

Linhas duplicadas (Order+Product): 8

=== ESTATÍSTICAS DAS MÉTRICAS ===
          Sales   Profit  Discount  Quantity
count   9994.00  9994.00   9994.00   9994.00
mean     229.86    28.66      0.16      3.79
std      623.25   234.26      0.21      2.23
min        0.44 -6599.98      0.00      1.00
25%       17.28     1.73      0.00      2.00
50%       54.49     8.67      0.20      3.00
75%      209.94    29.36      0.20      5.00
max    22638.48  8399.98      0.80     14.00

=== DISTRIBUIÇÃO SEGMENT ===
Segment
Consumer       5191
Corporate      3020
Home Office    1783
Name: count, dtype: int64

=== DISTRIBUIÇÃO REGION ===
Region
West       3203
East       2848
Central    2323
South      1620
Name: count, dtype: int64


In [ ]:
import pandas as pd, os
os.makedirs('../data/staging', exist_ok=True)
# Criar tabela de metas regionais (fonte legada simulada)
metas = pd.DataFrame({
'regiao' : ['Central','East','South','West'] * 3,
'categoria' : (['Furniture']*4 + ['Office Supplies']*4 + ['Technology']*4),
'meta_vendas' : [85000,120000,65000,140000,
95000,135000,72000,160000,
110000,145000,88000,175000],
'meta_lucro' : [12000,18000,9500,21000,
14000,20000,10800,24000,
16500,21750,13200,26250],

'ano_referencia': 2014,
})
os.makedirs('data/staging', exist_ok=True)
metas.to_csv('data/staging/metas_regionais.csv', index=False)

print('Arquivo metas_regionais.csv criado:')
print(metas.to_string(index=False))

Arquivo metas_regionais.csv criado:
 regiao       categoria  meta_vendas  meta_lucro  ano_referencia
Central       Furniture        85000       12000            2014
   East       Furniture       120000       18000            2014
  South       Furniture        65000        9500            2014
   West       Furniture       140000       21000            2014
Central Office Supplies        95000       14000            2014
   East Office Supplies       135000       20000            2014
  South Office Supplies        72000       10800            2014
   West Office Supplies       160000       24000            2014
Central      Technology       110000       16500            2014
   East      Technology       145000       21750            2014
  South      Technology        88000       13200            2014
   West      Technology       175000       26250            2014
